# 25MML1020

Shivanshika Aagarwal

**Assignment 3**

**Structured Streaming in Spark**

Structured Streaming is a scalable and fault-tolerant stream processing engine built on top of Apache Spark’s SQL engine. It allows users to process real-time data streams using the same high-level APIs (DataFrame and Dataset) used for batch processing. The key idea is that a streaming computation is treated as an unbounded table, and queries are executed incrementally as new data arrives.


**Difference between Structured Streaming and Batch Processing**
1. Data Type
* Structured Streaming: Processes continuous, unbounded data streams
* Batch Processing: Processes finite, static datasets
2. Processing Style
* Structured Streaming: Incremental processing (real-time or near real-time)
* Batch Processing: One-time processing of complete data
3. Latency
* Structured Streaming: Low latency
* Batch Processing: High latency
4. Use Cases
* Structured Streaming: Real-time analytics, monitoring systems
* Batch Processing: Historical data analysis and reporting
5. Execution Model
* Structured Streaming: Micro-batch or continuous processing
* Batch Processing: Single batch job execution

**Key Features of Structured Streaming**

1. Unified API: Same APIs for batch and streaming processing.
2. Scalability: Automatically scales across clusters.
3. Fault Tolerance: Uses checkpointing and write-ahead logs.
4. Exactly-once Processing: Ensures no duplicate or missing data.
5. Event-time Processing: Handles late-arriving data using watermarks.
6. Integration: Works with multiple sources like Kafka, file systems, and sockets.
7. Incremental Query Execution: Processes only new data instead of reprocessing the entire dataset.

**Real-time Use Cases**

* Fraud detection in banking systems
* IoT sensor data processing
* Real-time log and event monitoring
* Live dashboards and analytics
* Soial media sentiment analysis
* Stock market data analysis

In [1]:
# Install PySpark
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split
import os
import time

# Step 1: Create Spark Session
spark = SparkSession.builder \
    .appName("StructuredStreamingWordCount") \
    .getOrCreate()

# Step 2: Create a directory for streaming input
input_dir = "/content/stream_data"
os.makedirs(input_dir, exist_ok=True)

# Step 3: Read streaming data from directory
lines = spark.readStream \
    .format("text") \
    .load(input_dir)

# Step 4: Transform data (split into words)
words = lines.select(
    explode(
        split(lines.value, " ")
    ).alias("word")
)

# Step 5: Word count aggregation
wordCounts = words.groupBy("word").count()

# Step 6: Write output to console
query = wordCounts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

# Step 7: Simulate streaming by adding files
for i in range(3):
    with open(f"{input_dir}/file_{i}.txt", "w") as f:
        f.write(f"hello world hello spark batch {i}")
    time.sleep(5)

# Step 8: Keep the query running briefly
query.awaitTermination(15)

query.stop()